### np.linalg.det

In [1]:
def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def _get_item(a, idx):
    cur = a
    for i in idx:
        cur = cur[i]
    return cur


def _build_nested(shape, getter, prefix=()):
    if len(shape) == 0:
        return getter(prefix)
    return [_build_nested(shape[1:], getter, prefix + (i,)) for i in range(shape[0])]


def _minor(mat, exclude_row, exclude_col):
    """Return the (n-1)x(n-1) matrix obtained by removing row `exclude_row` and col `exclude_col`."""
    rows = []
    for i, row in enumerate(mat):
        if i == exclude_row:
            continue
        new_row = [row[j] for j in range(len(row)) if j != exclude_col]
        rows.append(new_row)
    return rows


def _det_2x2(mat):
    # mat is 2x2
    return mat[0][0] * mat[0][1] - mat[1][0] * mat[1][1]


def _det_laplace_2d(mat):
    n = len(mat)
    if n == 0:
        return 1.0
    if n == 1:
        return mat[0][0]
    if n == 2:
        return _det_2x2(mat)

    # use laplace expansion along first row
    total = 0.0
    for j in range(n):
        minor = _minor(mat, 0, j)
        cofactor = ((-1) ** j) * _det_laplace_2d(minor)
        total += mat[0][j] * cofactor

    return total


def np_linalg_det(a):
    """
    Pure Python equivalent of np.linalg.det.
    Input `a` must be a square matrix or a stack of square matrices:
    - shape (..., M, M)
    Output: determinant(s) along the last two axes.
    """
    shape = get_shape(a)
    ndim = len(shape)

    if ndim == 0:
        # 0-D scalar
        return 1.0

    if ndim == 1:
        raise ValueError("matrix must be at least 2-D")

    m = shape[-1]
    if m != shape[-2]:
        raise ValueError("last two axes must be square")

    if ndim == 2:
        # 2-D matrix: just compute its det
        return _det_laplace_2d(a)

    # ndim >= 3: last two axes form (..., M, M) matrices
    batch_shape = shape[:-2]

    def getter(prefix):
        mat = _get_item(a, prefix)
        return _det_laplace_2d(mat)

    return _build_nested(batch_shape, getter)

### test cases 

In [2]:
print(np_linalg_det([[1, 2], [3, 4]]))
print(np_linalg_det([[2, 3], [1, 4]]))
print(np_linalg_det([[1, 0], [0, 1]]))
print(np_linalg_det([[1, 0], [0, -1]]))
print(np_linalg_det([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))  # 3x3
print(np_linalg_det([[[1, 2], [3, 4]], [[1, 0], [0, 1]]]))  # stack of 2x2
print(np_linalg_det([[[1, 2], [3, 4]], [[1, 3], [2, 1]]]))

-10
2
0
0
-72.0
[-10, 0]
[-10, 1]


In [3]:
print(np_linalg_det([[1, 2], [3]])) 

ValueError: Jagged array: first row shape (2,), but row 1 shape (1,)

In [4]:
print(np_linalg_det([[1, 2], [3, 4], [5, 6]])) 

ValueError: last two axes must be square

In [6]:
print(np_linalg_det([[1, 2, 3], [4, 5, 6]]))

ValueError: last two axes must be square

In [8]:
print(np_linalg_det([1, 2])) 

ValueError: matrix must be at least 2-D